In [ ]:
import pandas as pd
import numpy as np
import blinksear as ear
import blinksdistance as distance
import blinkscolors as colors
import json
import matplotlib.pyplot as plt
from pathlib import Path

from blinkscorrelation import Correlation


In [ ]:
def generate_webgazer_dataframe(df):
    df_webgazer = df[~df["webgazer_data"].isna()]
    dfs = []
    for row in range(len(df_webgazer)):
        data_list = json.loads(df_webgazer["webgazer_data"].iloc[row])
        df_tmp = pd.DataFrame(data_list)
        df_tmp["trial"] = row + 1
        dfs.append(df_tmp)
    return pd.concat(dfs)

In [ ]:
# This can load all data but is not storing df_webgazer in any place
files = Path("./data").glob("*.csv")
dfs = []
i = 0
for file in files:
    if "blink-experiment" in str(file):
        print(file)
        df = pd.read_csv(file)
        df_tmp = generate_webgazer_dataframe(df)
        print(len(df_tmp))
        df_tmp["experiment"] = i
        df_tmp["file_name"] = str(file)
        i+=1
        dfs.append(df_tmp)

df_webgazer = pd.concat(dfs)
print(len(df_webgazer))

In [ ]:
blinkCorrrelation = Correlation()

df_webgazer["isBlinkCorrelation"] = df_webgazer.apply(
    lambda row: blinkCorrrelation.isBlinkByCorrelation(
        list(row["leftEye"]["data"].values()), list(row["rightEye"]["data"].values())
    ),
    axis=1,
)
df_webgazer

In [ ]:
df_webgazer["Blink_sample"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[0]==True))
df_webgazer["Blink_sample1"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[1]==True))
df_webgazer["Blink_sample2"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[2]==True))
df_webgazer["Blink_sample3"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[3]==True))
df_webgazer["Blink_sample4"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[4]==True))
df_webgazer["Blink_sample5"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[5]==True))
df_webgazer["Blink_sample6"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[6]==True))
df_webgazer["Blink_sample7"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[7]==True))
df_webgazer["Blink_sample8"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[8]==True))
df_webgazer["Blink_sample9"] = df_webgazer["isBlinkCorrelation"].apply(lambda x: int(x[9]==True))

In [ ]:
df_webgazer["Blink_sample0_diff"] = (df_webgazer['Blink_sample'].shift(1)+1 == df_webgazer['Blink_sample'])
df_webgazer["Blink_sample1_diff"] = (df_webgazer['Blink_sample1'].shift(1)+1 == df_webgazer['Blink_sample1'])
df_webgazer["Blink_sample2_diff"] = (df_webgazer['Blink_sample2'].shift(1)+1 == df_webgazer['Blink_sample2'])
df_webgazer["Blink_sample3_diff"] = (df_webgazer['Blink_sample3'].shift(1)+1 == df_webgazer['Blink_sample3'])
df_webgazer["Blink_sample4_diff"] = (df_webgazer['Blink_sample4'].shift(1)+1 == df_webgazer['Blink_sample4'])
df_webgazer["Blink_sample5_diff"] = (df_webgazer['Blink_sample5'].shift(1)+1 == df_webgazer['Blink_sample5'])
df_webgazer["Blink_sample6_diff"] = (df_webgazer['Blink_sample6'].shift(1)+1 == df_webgazer['Blink_sample6'])
df_webgazer["Blink_sample7_diff"] = (df_webgazer['Blink_sample7'].shift(1)+1 == df_webgazer['Blink_sample7'])
df_webgazer["Blink_sample8_diff"] = (df_webgazer['Blink_sample8'].shift(1)+1 == df_webgazer['Blink_sample8'])
df_webgazer["Blink_sample9_diff"] = (df_webgazer['Blink_sample9'].shift(1)+1 == df_webgazer['Blink_sample9'])

results = []
for x in range(0,10):
    tmp_result = df_webgazer.groupby(['trial', 'file_name'])['Blink_sample' + str(x) + '_diff'].sum().reset_index()
    #tmp_result['sample'] = x
    results.append(tmp_result)

cleaned_results = [results[0]] + [df.drop(columns=['trial', 'file_name']) for df in results[1:]]
result = pd.concat(cleaned_results, axis=1)
result

In [ ]:
threshold_columns = [col for col in result.columns if col not in ['trial', 'file_name']]
result['should_be'] = result['trial'].apply(lambda x: 2 if x == 3 else 10)
result[result['trial']==3]

In [ ]:
threshold_columns = result.columns[2:-1]
threshold_columns

In [ ]:
metodo = "CORRELACION"

# Obtener las columnas de thresholds (excluyendo 'trial' y 'Should be')
threshold_columns = result.columns[2:-1]

threshold_values = ["0,78-0,85", "0,8-0,9",	"0,7-0,8", "0,6-0,8", "0,7-0,9", "0,65-0,85", "0,71-0,81", "0,71-0,79", "0,73-0,77", "0,68-0,82"]
results_global = []
for i, threshold in enumerate(threshold_columns):
    preds = result[threshold]
    truth = result['should_be']
    mae = np.mean(np.abs(preds - truth))
    rmse = np.sqrt(np.mean((preds - truth) ** 2))
    results_global.append({'Threshold': threshold_values[i], 'Error Absoluto Medio': mae, 'Error Cuadrático Medio': rmse})

results_global_df = pd.DataFrame(results_global).sort_values(by='Error Cuadrático Medio')

results_global_df.head()



In [ ]:
metodo = "COLORES"

# Configuración de estilo
#sns.set(style="whitegrid")
plt.figure(figsize=(14, 7))
plt.style.use('bmh')
plt.rcParams['font.size'] = 16


# Calcular MAE y RMSE para cada threshold
metrics = []
for i, col in enumerate(threshold_columns):
    errors = result[col] - result['should_be']
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    metrics.append({'Threshold': threshold_values[i], 'EAM': mae, 'ECM': rmse})

# Crear DataFrame con los resultados
metrics_df = pd.DataFrame(metrics).sort_values(by='ECM')

# Gráfico combinado (MAE y RMSE)
plt.figure(figsize=(14, 7))
bar_width = 0.35
x = np.arange(len(metrics_df))

# Barras para MAE
bars_mae = plt.bar(x - bar_width/2, metrics_df['EAM'], width=bar_width, 
                   color='skyblue', label='EAM', edgecolor='grey')

# Barras para RMSE
bars_rmse = plt.bar(x + bar_width/2, metrics_df['ECM'], width=bar_width, 
                    color='salmon', label='ECM', edgecolor='grey')

# Añadir etiquetas y formato
plt.title(metodo + ' Comparación de EAM y ECM por Threshold', fontsize=18)
plt.xlabel('Threshold', fontsize=22)
plt.ylabel('Error (en cantidad de pestaneos)', fontsize=18)
plt.xticks(x, metrics_df['Threshold'], rotation=45, ha='right')
plt.legend(fontsize=18)

# Etiquetas con valores
for bar in bars_mae + bars_rmse:
    height = bar.get_height()
    plt.annotate(f'{height:.1f}',
                 (bar.get_x() + bar.get_width() / 2, height),
                 ha='center', va='bottom',
                 xytext=(0, 3),
                 textcoords='offset points')

# Resaltar el mejor threshold (MAE más bajo)
best_idx = metrics_df['ECM'].idxmin()
plt.gca().get_xticklabels()[0].set_color('red')
bars_mae[0].set_hatch('//')  # Patrón para resaltar
bars_rmse[0].set_hatch('//')  # Patrón para resaltar


plt.tight_layout()
plt.show()

# Mostrar tabla de resultados
print("\nResultados detallados:")
print(metrics_df.head().to_string(index=False))

In [ ]:

# Filtrar columnas que contienen valores de thresholds
# Calcular MAE y RMSE

# Convertir a DataFrame y ordenar
results_df = pd.DataFrame(results_global).sort_values(by='Error Absoluto Medio')

# Graficar
plt.figure(figsize=(14, 6))
plt.plot(results_global['Threshold'], results_df['Error Absoluto Medio'], marker='o', label='MAE')
plt.plot(results_global['Threshold'], results_df['Error Cuadrático Medio'], marker='s', label='RMSE')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Threshold Range')
plt.ylabel('Error')
plt.title('Error (MAE y RMSE) vs. Threshold Range')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
metrics_df.head()